# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a walkthrough for loading, exploring, and analyzing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully described in the FAIR² data package.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`. The metadata describes the structure, record sets, and fields available in the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL.
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

We will enumerate all record sets and display their structure using the `@id` for each. These IDs are crucial for referencing particular entities in subsequent operations.

In [ ]:
# Print all record sets and their fields with their @id

record_sets = list(dataset.record_sets)

print(f"There are {len(record_sets)} record sets in this dataset.\n")
for rs in record_sets:
    print(f"RecordSet: name={rs.name}, @id={rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id}, type={field.data_type})")
    print()

# Choose a record set to show example records (here, select the first one for demonstration)
if record_sets:
    example_record_set_id = record_sets[0].id
    print(f"\nSample records from RecordSet @id={example_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. All references are made using the record set and field `@id`s.

In [ ]:
# Extract all record sets to DataFrames, keyed by @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for rs_id in record_set_ids:
    # Convert records iterator to list of dicts
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded RecordSet {rs_id}. Shape: {df.shape}")
    else:
        print(f"RecordSet {rs_id} contains no records.")

# Display columns and first 5 rows from the primary record set (using the first one for demo)
if record_set_ids[0] in dataframes:
    main_rs_id = record_set_ids[0]
    print(f"Columns in RecordSet {main_rs_id}:\n", dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps, such as filtering records by field values, normalizing numeric fields, or grouping the data for further statistical analysis.

> All fields are referenced by their `@id`. Let's select a numeric field and a group field from the main record set for demonstration.

In [ ]:
# Example: Let's choose a numeric and a group field by their @id from the first record set

# Inspect the main record set's fields
main_rs = next((rs for rs in dataset.record_sets if rs.id == main_rs_id), None)

if main_rs:
    # Find the @id of a numeric field (e.g., GAD-7 score) and a group field (e.g., gender or village)
    num_field = None
    group_field = None
    for field in main_rs.fields:
        # Attempt to use a GAD/PHQ/PSQ score field if present, otherwise use the first integer/float
        if field.data_type in ('Integer', 'Float') and (num_field is None):
            num_field = field.id
        if field.name.lower() in ('gender', 'village') and group_field is None:
            group_field = field.id
    if num_field is None:
        print('No numeric field found for demonstration.')

    # For demonstration, print the chosen field IDs
    print(f'Numeric field selected: {num_field}')
    print(f'Group field selected: {group_field}')

    df = dataframes[main_rs_id]
    if num_field in df.columns:
        # Filter for values > 10 in the numeric field
        threshold = 10
        filtered_df = df[df[num_field] > threshold].copy()
        print(f"Filtered records where {num_field} > {threshold} (n={len(filtered_df)}):")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{num_field}_normalized"] = (filtered_df[num_field] - filtered_df[num_field].mean()) / filtered_df[num_field].std()
        print(f"\nNormalized values of {num_field}:")
        print(filtered_df[[num_field, f"{num_field}_normalized"]].head())

        # Group by the group_field if available
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped means by {group_field}:")
            print(grouped_df[[num_field, f"{num_field}_normalized"]].head())
    else:
        print(f'Numeric field {num_field} not found in DataFrame columns.')
else:
    print('Main record set not found.')

## 5. Visualization
Visualize data distributions or relationships between variables in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if num_field and num_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[num_field].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {num_field}')
    plt.xlabel(num_field)
    plt.ylabel('Count')
    plt.show()

# Boxplot of numeric field by group field, if available
if num_field and group_field and group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field], y=df[num_field])
    plt.title(f'{num_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(num_field)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored a Croissant FAIR² dataset about mental health survey data in Kilifi County, Kenya. Using the `mlcroissant` library, we:
- Loaded dataset metadata and printed the record set and field structure (`@id`s for reference)
- Loaded each record set into DataFrames for analysis
- Performed basic EDA, including field selection, filtering, normalization, and grouping
- Created visualizations to summarize the numeric data distributions

For in-depth analysis, refer to the original field documentation (using their `@id`s) and the Croissant schema for semantic meanings.